# M18 — Decide What Data Can Tell You

**Mission:** distinguish a repeatable signal from sampling noise without turning one statistic into certainty. The case is a fictional checkout experiment: variant B has a higher observed conversion rate than A, but the population effect is unknown.

## Runtime and inference contract

This lab is CPU-only, offline, deterministic where practical, and uses only committed synthetic fixtures plus the Python standard library. The pre-specified primary metric is checkout conversion; the estimand is **variant B rate minus variant A rate**; the comparison family contains one confirmatory comparison. Report the point estimate, interval, effect size, sample size, and assumptions together.

## Prediction-before-action protocol

Before each section marked **Predict before running**, pause and record a timestamped prediction in your separate learner notes. Include a direction, rough magnitude or range, and reason. Only then run the next cell. Compare observation with prediction and revise your model. The source notebook intentionally contains no learner answers.

In [ ]:
from pathlib import Path
import math
import statistics
import sys

START = Path.cwd()
candidates = [candidate for candidate in [START, *START.parents] if (candidate / 'datasets' / 'M18').is_dir()]
if candidates:
    ROOT = candidates[0]
else:
    if not candidates:
        raise FileNotFoundError('Run this notebook from inside the LearningOS-AI repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from simulations.M18.statistical_inference import (
    aggregate_counts,
    ascii_histogram,
    bootstrap_differences,
    bootstrap_interval,
    cherry_pick_null_comparisons,
    describe,
    difference_in_means,
    difference_standard_error,
    effect_sizes,
    expand_binary_outcomes,
    load_confounding_fixture,
    load_daily_experiment,
    normal_confidence_interval,
    pearson_correlation,
    permutation_test,
    simulate_familywise_false_positive_rate,
    simulate_sample_means,
    z_test_difference,
)

EXPERIMENT_PATH = ROOT / 'datasets' / 'M18' / 'checkout_experiment_daily.csv'
CONFOUNDING_PATH = ROOT / 'datasets' / 'M18' / 'seasonal_correlation.csv'
print(f'Repository root: {ROOT}')

In [ ]:
rows = load_daily_experiment(EXPERIMENT_PATH)
counts = aggregate_counts(rows)
outcomes_a = expand_binary_outcomes(rows, 'A')
outcomes_b = expand_binary_outcomes(rows, 'B')

print('Committed synthetic fixture totals:')
for group in ('A', 'B'):
    item = counts[group]
    print(f"  {group}: {item['conversions']}/{item['sessions']} = {item['rate']:.1%}")
print(f"Observed B − A: {difference_in_means(outcomes_a, outcomes_b):+.1%}")

## 1. Describe before inferring

**Predict before running:** For binary session outcomes, what will the mean, median, variance, and range say? Will the median distinguish 12% from 14% conversion? Then predict how the distribution of daily rates differs from the distribution of individual 0/1 outcomes.

In [ ]:
print('Individual binary outcomes')
print('A:', describe(outcomes_a))
print('B:', describe(outcomes_b))
print('\nThe median is zero in both groups: for a binary outcome, the mean is the rate,')
print('while the full 0/1 distribution and sample size explain what that rate summarizes.')

In [ ]:
daily_a = [row['conversions'] / row['sessions'] for row in rows if row['group'] == 'A']
daily_b = [row['conversions'] / row['sessions'] for row in rows if row['group'] == 'B']
print('Daily A rates:', describe(daily_a))
print('Daily B rates:', describe(daily_b))
print('\nA daily-rate distribution')
print(ascii_histogram(daily_a, bins=4, width=20))
print('\nB daily-rate distribution')
print(ascii_histogram(daily_b, bins=4, width=20))

**Explain:** Mean, median, and variance are not interchangeable. Their usefulness depends on the measurement scale and distribution. Daily rows are useful for inspection, but the unit of randomization and dependence structure—not whichever row format is convenient—must determine the inferential unit.

## 2. Sampling variation and standard-error intuition

**Predict before running:** Imagine a known population conversion probability of 13%. Across 2,000 repeated samples, where will sample rates center for n=25 and n=400? Predict each distribution's spread and their spread ratio.

In [ ]:
population_rate = 0.13
means_n25 = simulate_sample_means(population_rate, 25, repetitions=2_000, seed=1801)
means_n400 = simulate_sample_means(population_rate, 400, repetitions=2_000, seed=1802)

print('Repeated estimates, n=25:', describe(means_n25))
print('Repeated estimates, n=400:', describe(means_n400))
print('\nn=25 sampling distribution')
print(ascii_histogram(means_n25, bins=10, width=25))
print('\nn=400 sampling distribution')
print(ascii_histogram(means_n400, bins=10, width=25))

In [ ]:
empirical_se_25 = statistics.stdev(means_n25)
empirical_se_400 = statistics.stdev(means_n400)
theoretical_se_25 = math.sqrt(population_rate * (1 - population_rate) / 25)
theoretical_se_400 = math.sqrt(population_rate * (1 - population_rate) / 400)
print(f'Empirical SE, n=25:  {empirical_se_25:.4f} (theory {theoretical_se_25:.4f})')
print(f'Empirical SE, n=400: {empirical_se_400:.4f} (theory {theoretical_se_400:.4f})')
print(f'Spread ratio n=25 / n=400: {empirical_se_25 / empirical_se_400:.2f}')
print('Theory predicts sqrt(400/25) = 4.00.')

The sample estimate changes even though the population rate does not. Standard error describes this repeated-sample spread. Increasing n reduces random noise approximately with `1/sqrt(n)`; it does **not** remove selection bias, broken instrumentation, dependence, or confounding.

## 3. Estimate first; attach uncertainty

**Predict before running:** The observed difference is +2 percentage points. Predict the standard error and whether a 95% normal interval will include zero. What positive and negative effects might remain compatible with these data?

In [ ]:
observed_difference = difference_in_means(outcomes_a, outcomes_b)
observed_se = difference_standard_error(outcomes_a, outcomes_b)
analytic_interval = normal_confidence_interval(outcomes_a, outcomes_b)
print(f'B − A estimate: {observed_difference:+.3%}')
print(f'Estimated standard error: {observed_se:.3%}')
print(f'95% normal interval: [{analytic_interval[0]:+.3%}, {analytic_interval[1]:+.3%}]')

A frequentist 95% confidence procedure captures the fixed population effect in 95% of repeated experiments under its assumptions. It does not mean there is a 95% posterior probability that this already-computed interval contains the parameter. Because zero is compatible here, the data do not resolve the direction at the 95% level; that is not proof of equivalence or no effect.

## 4. Bootstrap/resampling

**Predict before running:** If A and B are independently resampled 4,000 times, predict the center and 95% percentile interval of the bootstrap differences. Will increasing resamples shrink uncertainty caused by having only 600 sessions per group?

In [ ]:
bootstrap_draws = bootstrap_differences(
    outcomes_a, outcomes_b, resamples=4_000, seed=1818
)
bootstrap_ci = bootstrap_interval(
    outcomes_a, outcomes_b, resamples=4_000, seed=1818
)
print('Bootstrap difference summary:', describe(bootstrap_draws))
print(f'95% percentile interval: [{bootstrap_ci[0]:+.3%}, {bootstrap_ci[1]:+.3%}]')

In [ ]:
print(ascii_histogram(bootstrap_draws, bins=12, width=30))
print('\nMore resamples reduce Monte Carlo jitter; they do not create more observed sessions.')

The bootstrap treats the observed samples as stand-ins for their populations. It is useful for approximating the estimator's sampling distribution, but it cannot repair non-random assignment, duplicated users, interference, a broken metric, missing-not-at-random outcomes, or a sample that misses the target population.

## 5. Effect size before threshold

**Predict before running:** Convert +2 percentage points into relative lift and number needed to expose for one additional conversion if the point estimate repeated. Which scale best supports an operational decision?

In [ ]:
effects = effect_sizes(outcomes_a, outcomes_b)
print(f"Absolute risk difference: {effects['risk_difference']:+.1%}")
print(f"Relative risk: {effects['relative_risk']:.3f}")
print(f"Relative lift: {effects['relative_lift']:+.1%}")
print(f"Point-estimate NNT/exposures per additional conversion: {effects['number_needed_to_treat']:.1f}")
print('Absolute and relative descriptions are both accurate but answer different questions.')

## 6. A useful hypothesis test

Confirmatory null: A and B have the same conversion probability. Statistic: absolute difference in rates. **Predict before running:** Will pooled-normal and permutation p-values fall below 0.05? State why labels are exchangeable under the permutation null—and how failed randomization or repeated-user dependence would break that argument.

In [ ]:
z_result = z_test_difference(outcomes_a, outcomes_b)
print(f"Pooled-null z: {z_result['z_score']:.3f}")
print(f"Two-sided normal p-value: {z_result['p_value']:.4f}")

In [ ]:
permutation_result = permutation_test(
    outcomes_a, outcomes_b, permutations=4_000, seed=1819
)
print(permutation_result)

The p-value is the probability, under the specified null procedure, of a statistic at least as extreme as observed. It is **not** the probability that the null is true, the probability the result occurred “by chance,” or an effect size. Here it supports “unresolved at this precision,” not “the variants are equal.” A product decision must also use the interval, minimum useful effect, downside, cost of delay, and validity checks.

## 7. Correlation is not a causal identification strategy

**Predict before running:** Predict the signs and rough sizes of correlations among temperature, ice-cream sales, and drownings. Name the common seasonal cause and explain why a high sales/drownings correlation alone cannot identify an intervention effect.

In [ ]:
seasonal = load_confounding_fixture(CONFOUNDING_PATH)
temperature = [row['temperature_c'] for row in seasonal]
sales = [row['ice_cream_sales'] for row in seasonal]
drownings = [row['drownings'] for row in seasonal]
correlations = {
    'sales_vs_drownings': pearson_correlation(sales, drownings),
    'temperature_vs_sales': pearson_correlation(temperature, sales),
    'temperature_vs_drownings': pearson_correlation(temperature, drownings),
}
for name, value in correlations.items():
    print(f'{name}: {value:.3f}')

The fixture is constructed so hotter weather raises both ice-cream demand and swimming exposure. Sales do not therefore cause drownings. A causal claim needs a defensible design or identification strategy—such as randomization, a credible natural experiment, or justified adjustment with temporal and domain evidence—not a larger correlation coefficient.

## 8. Controlled failure: cherry-pick across multiple comparisons until “significant”

Every generated comparison below is a true null. A bad workflow runs 20, keeps only `p < 0.05`, invents a story matching the surviving signs, and hides the rest. **Predict before running:** What is the chance that at least one null p-value falls below 0.05? Will any seeded nominal win pass a Bonferroni threshold of 0.05/20?

In [ ]:
failure = cherry_pick_null_comparisons(comparisons=20, seed=1800, alpha=0.05)
print('Misleading selected-only report:')
for result in failure['selected']:
    print(result)
print(f"\nHidden comparison count: {len(failure['comparisons']) - len(failure['selected'])}")
print(f"Minimum p-value: {failure['minimum_p_value']:.6f}")
print(f"Bonferroni threshold: {failure['bonferroni_threshold']:.6f}")

In [ ]:
print('Full audit trail (nothing hidden):')
for result in failure['comparisons']:
    marker = 'nominal-only' if result['p_value'] < failure['alpha'] else ''
    print(f"{result['metric']}: effect={result['estimated_effect']:+.3%}, p={result['p_value']:.5f} {marker}")
familywise_rate = simulate_familywise_false_positive_rate(
    comparisons=20, families=2_000, seed=1820, alpha=0.05
)
theoretical_familywise_rate = 1 - (1 - 0.05) ** 20
print(f'\nSimulated chance of >=1 false positive: {familywise_rate:.1%}')
print(f'Theory under independence: {theoretical_familywise_rate:.1%}')

In [ ]:
adjusted_passes = [
    result for result in failure['comparisons']
    if result['p_value'] < failure['bonferroni_threshold']
]
print('Results passing pre-declared Bonferroni control:', adjusted_passes)
print('Repair: pre-specify the primary metric, estimand, comparison family, and stopping rule;')
print('retain every analysis; label exploration; and confirm new hypotheses on fresh data.')

Multiplicity correction addresses a declared statistical error target; it does not repair biased samples, confounding, measurement failures, silent exclusions, or an outcome definition changed after inspection. Bonferroni is transparent and strict, not universally optimal. Choose familywise-error or false-discovery-rate control to match the actual decision and record that choice before results.

## 9. Code reading and assumption audit

Before rereading outputs, manually trace `bootstrap_differences`, `permutation_test`, and `cherry_pick_null_comparisons` using `missions/M18/code_reading.md`. Identify the resampling unit, mutable random state, null-world transformation, selection boundary, and one assumption each function cannot verify.

In [ ]:
# Executable invariants: a Restart + Run All fails loudly if the lesson drifts.
assert counts['A']['sessions'] == 600 and counts['A']['conversions'] == 72
assert counts['B']['sessions'] == 600 and counts['B']['conversions'] == 84
assert math.isclose(observed_difference, 0.02, abs_tol=1e-12)
assert analytic_interval[0] < 0 < analytic_interval[1]
assert bootstrap_ci[0] < 0 < bootstrap_ci[1]
assert z_result['p_value'] > 0.05
assert permutation_result['p_value'] > 0.05
assert empirical_se_400 < empirical_se_25
assert correlations['sales_vs_drownings'] > 0.9
assert failure['minimum_p_value'] < 0.05
assert adjusted_passes == []
assert 0.55 < familywise_rate < 0.75
print('All M18 runtime invariants passed.')

## 10. Decision worksheet (complete outside the source notebook)

Record: population and sample; unit of randomization and analysis; metric and estimand; observed distribution; point estimate and units; analytic and bootstrap intervals; absolute and relative effect; null and p-value; comparison family and stopping rule; assignment/dependence/measurement checks; minimum useful effect; costs of false positive and false negative; current decision; evidence that would change it.

Then complete `missions/M18/adr_prompt.md` and the no-AI transfer gate.

## What these data can—and cannot—say

The sample says B's observed conversion rate is 2 percentage points higher. Under the stated assumptions, both analytic and bootstrap uncertainty ranges include zero, and the planned tests do not resolve a difference at the 0.05 level. Effects large enough to matter remain compatible with the data, so “no effect” is also too strong. The defensible state is **unresolved: combine operational stakes with more evidence or a pre-specified follow-up**. Nothing here validates assignment, independence, measurement quality, generalization, or causality automatically.